<h2 align="center">Divide Data<h2>

In [2]:
import os
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import numpy as np

In [3]:
# load the data
data = pd.read_csv('../Data Collection and Annotation/Data.csv')

# Fine Grained NE

In [11]:
# save the data to a tsv file
with open('Data.tsv', 'wt') as file:
    writer = csv.writer(file, delimiter='\t')
    writer.writerow(['Sentence#','Token','Segment','Tag-Mixed'])
    for i in range(0,len(data)):
        if(data['Tag'][i] == "MIXED"):
            segment = str(data['Segment'][i]).replace(","," ")
            mixedTag = str(data['Tag-Mixed'][i]).replace(","," ")
        else:
            segment = str(data['Segment'][i])
            mixedTag = str(data['Tag-Mixed'][i])
        writer.writerow([data['Sentence#'][i],data['Token'][i],segment,mixedTag])

In [12]:
# this function to save any data into txt file in this formate "token \t segment \t tag"
def saveToFile(data,fileName):
    Str = ""
    for sentence in data:
        for token, segment, tag in sentence:
            Str += (str(token+"\t"+segment+"\t"+tag+"\n"))
        Str += "\n"

    file = open(fileName, "w")
    file.write(Str)
    file.close()

In [33]:
os.makedirs('Train/5Fold_Evaluation/Dev', exist_ok=True)
os.makedirs('Train/5Fold_Evaluation/Train', exist_ok=True)
os.makedirs('Test', exist_ok=True)
os.makedirs('Full_Sentence_TXT', exist_ok=True)

In [34]:
# load the tsv file containing all data
workingData = pd.read_table(os.getcwd() + '/Data.tsv')

# extract the (token,segment,tag) from the data
agg_func = lambda s: [(w, p, t) for w, p, t in zip(s["Token"].values.tolist(),s["Segment"].values.tolist(),s["Tag-Mixed"].values.tolist())]

# grouped the data with its sentence to separate the sentence by a newline
grouped = workingData.groupby("Sentence#").apply(agg_func)
sentences = [s for s in grouped]

# save all values to one file
saveToFile(sentences,"all.aren")

# split the sentences into train and test sets by 800:229 ratio
train, test = train_test_split(sentences, test_size=0.222546161, random_state=2018)

# save the train and test sets
saveToFile(train,"Train/train.aren")
saveToFile(test,"Test/test.aren")

# apply 5-fold evaluation matrix and save its results
trainAndDev = np.array(train)
kfold = KFold(5) # 5 split
i = 1
for train, dev in kfold.split(trainAndDev):
    saveToFile(trainAndDev[train],"Train/5Fold_Evaluation/Train/train."+str(i)+".aren")
    saveToFile(trainAndDev[dev],"Train/5Fold_Evaluation/Dev/dev."+str(i)+".aren")
    i += 1


/home/innovator/.local/lib/python3.6/site-packages/ipykernel_launcher.py:22: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray


In [35]:
# make sure that every segment in the same length of its token
def checkEqualLen(workingData):
    for i in range(0,len(workingData)):
        token = workingData["Token"][i]
        segment = workingData["Segment"][i].split(" ")
        s = ""
        if(len(segment) >0):
            for j in range(0,len(segment)):
                s += segment[j]
        else:
            s = len(segment)
        if(len(token) != len(s)):
            print(i,token)

checkEqualLen(workingData)

In [36]:
def convertIntoTXT(input_file,output_file):
    file  = open(input_file, 'r').read().split("\n\n")
    file_out = open(output_file, 'w')
    for sen in file:
        tokens = sen.split("\n")
        sentence = ""
        for token in tokens:
            sentence += str(token.split("\t")[0])+" "
        sentence += "\n"
        file_out.writelines(sentence)
    file_out.close() 

convertIntoTXT("Train/5Fold_Evaluation/Train/train.1.aren","Full_Sentence_TXT/train.txt")
convertIntoTXT("Train/5Fold_Evaluation/Dev/dev.1.aren","Full_Sentence_TXT/dev.txt")
convertIntoTXT("Test/test.aren","Full_Sentence_TXT/test.txt")

# Coarse Grained NE

In [8]:
# load the data
data = pd.read_csv('../Data Collection and Annotation/Data.csv')

In [5]:
data

,#,Sentence#,Token,Tag,Tag-Mixed,Segment,SegmentLen
0,1,1,derasa,AR,AR,derasa,6
1,2,1,?,OTHER,OTHER,?,1
2,3,1,ya,AR,AR,ya,2
3,4,1,addy,NE.AR,NE.AR,addy,4
4,5,1,howa,AR,AR,howa,4
...,...,...,...,...,...,...,...
30316,30317,2507,kaman,AR,AR,kaman,5
30317,30318,2507,f,AR,AR,f,1
30318,30319,2507,mafeesh,AR,AR,mafeesh,7
30319,30320,2507,habd,AR,AR,habd,4


In [17]:
data["Tag"].replace({"NE.AR": "NE", "NE.EN": "NE"}, inplace=True)
data["Tag-Mixed"].replace({"NE.AR": "NE", "NE.EN": "NE"}, inplace=True)

In [18]:
data

,#,Sentence#,Token,Tag,Tag-Mixed,Segment,SegmentLen
0,1,1,derasa,AR,AR,derasa,6
1,2,1,?,OTHER,OTHER,?,1
2,3,1,ya,AR,AR,ya,2
3,4,1,addy,NE,NE,addy,4
4,5,1,howa,AR,AR,howa,4
...,...,...,...,...,...,...,...
30316,30317,2507,kaman,AR,AR,kaman,5
30317,30318,2507,f,AR,AR,f,1
30318,30319,2507,mafeesh,AR,AR,mafeesh,7
30319,30320,2507,habd,AR,AR,habd,4


In [20]:
data.to_csv('./CoarseGrainedNE_data.csv')